# Reflection as a Cyclic Graph

**What you will build:** the reflection loop from 0.5 (generate -> critique -> improve) as a LangGraph **cycle**. This is where a graph beats both plain Python and n8n: the loop is a first-class, drawable structure with a clean stop condition. Maps to chapter 05 of the n8n course.

> **Run it:** open in [Google Colab](https://colab.research.google.com/github/ezponda/ai-agents-course/blob/main/courses/python_code/book/25_reflection_cyclic_graph.ipynb) (nothing to install), or run locally in Jupyter. Each notebook installs what it needs in its first cell.

In [ ]:
# Setup — LangChain + LangGraph on OpenRouter. Run once.
!pip install -q "langchain>=1.3,<2.0" "langgraph>=1.2,<2.0" "langchain-openai>=1.3,<2.0"

import os, getpass
from langchain_openai import ChatOpenAI

if not os.environ.get("OPENROUTER_API_KEY"):
    os.environ["OPENROUTER_API_KEY"] = getpass.getpass("Paste your OpenRouter API key: ")

MODEL_NAME = "meta-llama/llama-3.3-70b-instruct"
llm = ChatOpenAI(model=MODEL_NAME, base_url="https://openrouter.ai/api/v1",
                 api_key=os.environ["OPENROUTER_API_KEY"])
print("Ready:", MODEL_NAME)

## Generate, critique, loop

Two nodes and a conditional edge that can point *back*: `generate` writes a draft (using any feedback so far), `critique` judges it, and the router either loops back to `generate` or ends. We cap iterations so it always terminates.

In [ ]:
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END

class State(TypedDict):
    task: str
    draft: str
    feedback: str
    passed: bool
    iterations: int

def generate(state: State) -> dict:
    fb = state.get("feedback", "")
    prompt = f"Task: {state['task']}"
    if fb:
        prompt += f"\nRevise your previous attempt using this feedback: {fb}\nPrevious: {state['draft']}"
    r = llm.invoke([{"role": "system", "content": "You are a concise copywriter."},
                    {"role": "user", "content": prompt}])
    return {"draft": r.content, "iterations": state.get("iterations", 0) + 1}

def critique(state: State) -> dict:
    r = llm.invoke([{"role": "system", "content":
                     "You are a strict editor. Reply exactly 'PASS' ONLY if the draft is exactly 3 sentences AND under "
                     "25 words total. "
                     "Otherwise reply with one sentence of specific feedback."},
                    {"role": "user", "content": state["draft"]}])
    verdict = r.content.strip()
    return {"passed": verdict.upper().startswith("PASS"), "feedback": verdict}

def route(state: State) -> str:
    if state["passed"] or state["iterations"] >= 3:   # stop condition
        return END
    return "generate"                                   # loop back to improve

In [ ]:
builder = StateGraph(State)
builder.add_node("generate", generate)
builder.add_node("critique", critique)
builder.add_edge(START, "generate")
builder.add_edge("generate", "critique")
builder.add_conditional_edges("critique", route, {"generate": "generate", END: END})

graph = builder.compile()

In [ ]:
from IPython.display import Image, display
try:
    display(Image(graph.get_graph().draw_mermaid_png()))
except Exception:
    print("Mermaid image unavailable, showing text instead:\n")
    print(graph.get_graph().draw_mermaid())

The drawn graph has a visible cycle `critique -> generate`. Run it and watch it iterate until the critic passes it or it hits the cap:

In [ ]:
out = graph.invoke({"task": "Write a product blurb for a noise-cancelling coffee mug."})
print("iterations:", out["iterations"])
print("passed:    ", out["passed"])
print("final:     ", out["draft"])

Same pattern as 0.5, but now the loop *is* the graph — inspectable, drawable, with the stop condition in one obvious place. Cycles are native to LangGraph: in n8n the loop-back needs an If node and careful wiring, whereas here it's a single edge — the reason a stateful graph earns its keep for this kind of flow.

```{note}
As in 0.5, a code check (`iterations >= 3`) guarantees termination, while the LLM critic handles the subjective quality call. Always give a cyclic graph a hard cap — it's the graph equivalent of `max_turns`.
```

**What's next:** in **2.6** we go from one agent to **many** — a supervisor that routes work to specialist agents.